# Aviação Civil Brasileira — Panorama 2019 x 2026 (ANAC)

**Objetivo:** responder 4 perguntas de negócio sobre o transporte aéreo regular de passageiros no Brasil, usando os microdados públicos da ANAC (Dados Estatísticos do Transporte Aéreo).

1. Como a demanda por transporte aéreo evoluiu entre 2000 e 2026?
2. Qual foi o tamanho da queda na pandemia e o ritmo da retomada?
3. Como está distribuído o mercado entre as companhias aéreas hoje vs. antes da pandemia?
4. O tráfego aéreo está mais ou menos concentrado em poucos aeroportos?

**Fonte:** [Dados Estatísticos do Transporte Aéreo — ANAC](https://www.gov.br/anac/pt-br/assuntos/dados-e-estatisticas/dados-estatisticos-do-transporte-aereo), consolidado em `Dados_Estatisticos.csv` (1.09M registros, 2000–2026).

**Nota de dados:** os 3 arquivos-fonte trazem uma linha de cabeçalho descartável (`Atualizado em: ...`) antes do header real, e `AerodromosPublicos.csv` está em `cp1252` (não `latin1`/`utf-8`) — ambos tratados na célula de carregamento abaixo.

**Nota metodológica:** 2026 possui dados apenas de janeiro a maio (dataset parcial). Todas as comparações "2019 x 2026" usam a janela jan–mai em ambos os anos, para evitar viés de sazonalidade — comparar ano completo com ano parcial inflaria artificialmente a queda. Esse recorte está documentado em cada função de agregação.

## 1. Carregamento e validação

Carregamos os 3 CSVs da ANAC. `Dados_Estatisticos.csv` traz só as 6 colunas necessárias (evita ~1,4GB de RAM das 38 colunas originais) e já filtrado para voos regulares de passageiros — a definição correta de "tráfego aéreo comercial" (exclui fretamento e voos improdutivos, que distorceriam a leitura de demanda). Os dois CSVs de dimensão (aeródromos e empresas) são pequenos e carregados por completo, para contexto/validação.

In [2]:
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path

DATA_DIR = Path(".")  # ajuste se os CSVs estiverem em outra pasta (ex.: Path("/content/drive/MyDrive/anac"))

In [3]:
!pip install -q -U kaleido==1.3.0

import kaleido

# Kaleido >= 1.0 depende de um Chrome headless próprio (não é mais "chrome-free" como o 0.1.0).
# Esta chamada baixa esse Chrome uma única vez por ambiente (necessário no Colab/máquina nova).
try:
    kaleido.get_chrome_sync()
    print("Chrome do Kaleido pronto.")
except Exception as exc:
    print(f"Aviso: download automático do Chrome falhou ({exc}).")
    print("Rode manualmente no terminal/célula: !kaleido_get_chrome")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 kB 3.1 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/kaleido/_sync_server.py:11: UserWarning: 


This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.

You can however, use the Kaleido API directly which will work with your plotly version. `kaleido.write_fig(...)`, for example. Please see the kaleido documentation.

  from .kaleido import Kaleido


Chrome do Kaleido pronto.


In [4]:
COLS_VOOS = [
    "EMPRESA_NOME", "ANO", "MES", "AEROPORTO_DE_ORIGEM_NOME",
    "GRUPO_DE_VOO", "PASSAGEIROS_PAGOS",
]


def carregar_voos(data_dir: Path = DATA_DIR) -> pd.DataFrame:
    """Carrega Dados_Estatisticos.csv já filtrado para voos regulares de passageiros.

    Lê apenas as 6 colunas necessárias para a análise (evita ~1,4GB de RAM
    desnecessária vindos das 38 colunas originais do arquivo).

    Args:
        data_dir: Diretório onde o CSV está localizado.

    Returns:
        DataFrame filtrado contendo apenas voos com GRUPO_DE_VOO == "REGULAR".

    Raises:
        FileNotFoundError: se o arquivo não existir no caminho informado.
        ValueError: se colunas essenciais estiverem ausentes após a leitura.
    """
    path = data_dir / "Dados_Estatisticos.csv"
    try:
        df = pd.read_csv(
            path, sep=";", encoding="utf-8-sig", skiprows=1,
            quotechar='"', usecols=COLS_VOOS, low_memory=False,
        )
    except FileNotFoundError as exc:
        raise FileNotFoundError(f"Arquivo não encontrado: {path}") from exc

    missing = set(COLS_VOOS) - set(df.columns)
    if missing:
        raise ValueError(f"Colunas ausentes no dataset: {missing}")

    return df[df["GRUPO_DE_VOO"] == "REGULAR"].copy()


def carregar_dimensoes(data_dir: Path = DATA_DIR) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Carrega as dimensões de apoio: aeródromos públicos e empresas aéreas ativas.

    Usadas para contexto e validação cruzada — as 4 perguntas de negócio do notebook
    são respondidas inteiramente a partir de Dados_Estatisticos.csv (via carregar_voos).

    Args:
        data_dir: Diretório onde os CSVs estão localizados.

    Returns:
        Tupla (df_aerodromos, df_empresas).

    Raises:
        FileNotFoundError: se algum dos arquivos não existir.
    """
    try:
        df_aerodromos = pd.read_csv(
            data_dir / "AerodromosPublicos.csv", sep=";", encoding="cp1252",
            skiprows=1, low_memory=False,
        )
        df_empresas = pd.read_csv(
            data_dir / "pda_empresas_aereas_nacionais.csv", sep=";", encoding="utf-8-sig",
            skiprows=1, quotechar='"', low_memory=False,
        )
    except FileNotFoundError as exc:
        raise FileNotFoundError(f"Arquivo de dimensão não encontrado: {exc.filename}") from exc

    return df_aerodromos, df_empresas


voos = carregar_voos()
df_aerodromos, df_empresas = carregar_dimensoes()

print(f"Voos regulares: {len(voos):,} registros | período: {voos['ANO'].min()}–{voos['ANO'].max()}")
print(f"Aeródromos públicos cadastrados: {len(df_aerodromos):,}")
print(f"Empresas aéreas ativas: {len(df_empresas):,}")

Voos regulares: 721,817 registros | período: 2000–2026
Aeródromos públicos cadastrados: 497
Empresas aéreas ativas: 729


In [5]:
# Paleta mínima — 1 cor de destaque, 1 neutra, 1 de alerta. Sem gradientes, sem 8 cores por gráfico.
COR_PRINCIPAL = "#1f4e79"
COR_DESTAQUE = "#c0392b"
COR_NEUTRA = "#b0b0b0"
TEMPLATE = "simple_white"

## 2. Evolução histórica da demanda (2000–2026)

**Pergunta de negócio:** a demanda por transporte aéreo é estrutural e crescente, ou instável?

In [6]:
import plotly.graph_objects as go

def serie_historica_anual(df: pd.DataFrame) -> pd.DataFrame:
    """Agrega passageiros pagos por ano.

    Args:
        df: DataFrame de voos regulares.

    Returns:
        DataFrame com colunas ["ANO", "PASSAGEIROS_PAGOS", "PARCIAL"].
        O último ano é sinalizado como parcial (jan-mai de 2026).
    """
    serie = df.groupby("ANO", as_index=False)["PASSAGEIROS_PAGOS"].sum().sort_values("ANO")
    ultimo_ano = serie["ANO"].max()
    serie["PARCIAL"] = serie["ANO"] == ultimo_ano
    return serie


def fig_evolucao_historica(serie_hist: pd.DataFrame) -> go.Figure:
    """Gera gráfico de linha da evolução anual, com 2026 marcado como parcial."""
    completos = serie_hist[~serie_hist["PARCIAL"]]
    parcial = serie_hist[serie_hist["PARCIAL"]]

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=completos["ANO"], y=completos["PASSAGEIROS_PAGOS"],
        mode="lines+markers", line=dict(color=COR_PRINCIPAL, width=2.5),
        marker=dict(size=5), hovertemplate="%{x}: %{y:,.0f} pax<extra></extra>",
    ))
    fig.add_trace(go.Scatter(
        x=parcial["ANO"], y=parcial["PASSAGEIROS_PAGOS"],
        mode="markers+text", marker=dict(size=10, color=COR_DESTAQUE, symbol="diamond"),
        text=["2026 (parcial, jan-mai)"], textposition="top center",
        hovertemplate="%{x} (parcial): %{y:,.0f} pax<extra></extra>",
    ))
    minimo = serie_hist.loc[serie_hist["PASSAGEIROS_PAGOS"].idxmin()]
    fig.add_annotation(
        x=minimo["ANO"], y=minimo["PASSAGEIROS_PAGOS"],
        text=f"2020: {minimo['PASSAGEIROS_PAGOS']/1e6:.0f}M pax (-57,6% vs 2019)",
        showarrow=True, arrowhead=2, ax=40, ay=-40,
    )
    fig.update_layout(
        template=TEMPLATE, title="Passageiros pagos por ano — voos regulares (2000-2026)",
        yaxis_title="Passageiros pagos", showlegend=False,
        margin=dict(t=60, b=40, l=60, r=40), height=420,
    )
    return fig


serie_hist = serie_historica_anual(voos)
fig1 = fig_evolucao_historica(serie_hist)
fig1.show()

**Leitura:** a demanda é estruturalmente crescente desde 2000, com uma única quebra abrupta (pandemia, 2020: -57,6% vs 2019). A retomada superou o patamar pré-pandemia em 2024. 2026 está marcado como parcial para não ser lido como queda.

## 3. Recuperação pós-pandemia

**Pergunta de negócio:** quanto tempo levou para o setor voltar ao patamar pré-covid, e em que ritmo o ano corrente está indo?

In [7]:
def recuperacao_pos_pandemia(df: pd.DataFrame) -> pd.DataFrame:
    """Compara volume de passageiros em 2019 (pré), 2020 (colapso) e 2025 (retomada plena — ano completo)."""
    anos_completos = df[df["ANO"].isin([2019, 2020, 2025])]
    serie = anos_completos.groupby("ANO", as_index=False)["PASSAGEIROS_PAGOS"].sum()
    base_2019 = serie.loc[serie["ANO"] == 2019, "PASSAGEIROS_PAGOS"].iloc[0]
    serie["DELTA_PCT_VS_2019"] = (serie["PASSAGEIROS_PAGOS"] / base_2019 - 1) * 100
    return serie


def ritmo_ano_corrente(df: pd.DataFrame) -> dict:
    """Compara jan-mai/2026 (parcial) contra o mesmo período de 2025 — comparação justa (mesma janela)."""
    janmai = df[df["MES"] <= 5]
    pax_2025 = janmai.loc[janmai["ANO"] == 2025, "PASSAGEIROS_PAGOS"].sum()
    pax_2026 = janmai.loc[janmai["ANO"] == 2026, "PASSAGEIROS_PAGOS"].sum()
    delta_pct = (pax_2026 / pax_2025 - 1) * 100 if pax_2025 else float("nan")
    return {"pax_2025_jan_mai": pax_2025, "pax_2026_jan_mai": pax_2026, "delta_pct": delta_pct}


def fig_recuperacao_pandemia(recup: pd.DataFrame, ritmo: dict) -> go.Figure:
    """Gera gráfico de barras: 2019 x 2020 x 2025, com callout do ritmo de 2026."""
    cores = {2019: COR_NEUTRA, 2020: COR_DESTAQUE, 2025: COR_PRINCIPAL}
    fig = go.Figure(go.Bar(
        x=recup["ANO"].astype(str), y=recup["PASSAGEIROS_PAGOS"],
        marker_color=[cores[a] for a in recup["ANO"]],
        text=[f"{v/1e6:.0f}M<br>({d:+.1f}%)" if a != 2019 else f"{v/1e6:.0f}M<br>(base)"
              for v, d, a in zip(recup["PASSAGEIROS_PAGOS"], recup["DELTA_PCT_VS_2019"], recup["ANO"])],
        textposition="outside",
    ))
    fig.add_annotation(
        xref="paper", x=1.0, y=1.15, yref="paper", showarrow=False, align="right",
        text=(f"Jan-mai/2026 vs jan-mai/2025: {ritmo['pax_2026_jan_mai']/1e6:.1f}M "
              f"({ritmo['delta_pct']:+.1f}%)"),
        font=dict(size=12, color=COR_PRINCIPAL),
    )
    fig.update_layout(
        template=TEMPLATE, title="Recuperação pós-pandemia — passageiros pagos (base 2019)",
        yaxis_title="Passageiros pagos", showlegend=False,
        margin=dict(t=90, b=40, l=60, r=40), height=420,
    )
    return fig


recup = recuperacao_pos_pandemia(voos)
ritmo = ritmo_ano_corrente(voos)
fig2 = fig_recuperacao_pandemia(recup, ritmo)
fig2.show()

**Leitura:** o setor levou 5 anos para superar o patamar pré-pandemia (2025 fechou 10,5% acima de 2019). O ritmo de 2026 (jan-mai) segue positivo, +7,5% vs o mesmo período de 2025 — indicando continuidade do crescimento, não estagnação.

## 4. Participação de mercado por companhia (jan-mai 2019 x 2026)

**Pergunta de negócio:** o mercado ficou mais concentrado nas mãos de poucas companhias desde a pandemia?

**Leitura:** as 3 maiores companhias concentram mais de 80% do mercado em ambos os anos — o oligopólio já existia antes da pandemia e não se alterou estruturalmente.

In [50]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

def fig_market_share(share: pd.DataFrame) -> go.Figure:
    """
    Gráfico executivo comparando o Market Share
    das companhias aéreas entre Jan–Mai de 2019 e 2026.
    """

    # =====================================================
    # Ordem das empresas
    # =====================================================

    ordem = share.attrs["ordem"]

    fig = go.Figure()

    # =====================================================
    # Barras
    # =====================================================

    for ano, cor, legenda in [

        (2019, COR_NEUTRA, "Pré-pandemia (2019)"),

        (2026, COR_PRINCIPAL, "Atual (2026)"),

    ]:

        sub = (
            share.loc[
                share["ANO"] == ano
            ]
            .set_index("EMPRESA_PLOT")
            .reindex(ordem)
            .reset_index()
        )

        textos = []

        for _, row in sub.iterrows():

            if ano == 2026:

                delta = row["DELTA_PP"]

                if delta > 0:
                    seta = "▲"
                elif delta < 0:
                    seta = "▼"
                else:
                    seta = "■"

                textos.append(
                    f"<b>{row['PCT']:.1f}%</b>"
                    f"<br>{seta} {delta:+.1f} p.p."
                )

            else:

                textos.append(
                    f"{row['PCT']:.1f}%"
                )

        fig.add_trace(

            go.Bar(

                y=sub["EMPRESA_PLOT"],

                x=sub["PCT"],

                orientation="h",

                name=legenda,

                marker=dict(

                    color=cor,

                    line=dict(

                        color="white",

                        width=1

                    )

                ),

                text=textos,

                textposition="outside",

                textfont=dict(

                    size=13

                ),

                customdata=sub["PASSAGEIROS"],

                hovertemplate=(

                    "<b>%{y}</b><br>"

                    "Participação: <b>%{x:.2f}%</b><br>"

                    "Passageiros: <b>%{customdata:,.0f}</b>"

                    "<extra></extra>"

                )

            )

        )

         # =====================================================
    # Layout Executivo
    # =====================================================

    fig.update_layout(

        template=TEMPLATE,

        barmode="group",

        height=920,

        plot_bgcolor="white",

        paper_bgcolor="white",

        title=dict(

            text=(
                "<b>Market Share das Companhias Aéreas Brasileiras</b>"
                "<br>"
                "<sup>Comparação Janeiro–Maio • Pré-pandemia (2019) × Atual (2026)</sup>"
            ),

            x=0.5,

            xanchor="center",

            font=dict(
                size=24
            )

        ),

        legend=dict(

            orientation="h",

            y=1.05,

            x=1,

            xanchor="right",

            yanchor="bottom",

            font=dict(
                size=13
            )

        ),

        xaxis=dict(

            title="% dos passageiros pagos",

            showgrid=True,

            gridcolor="#EAEAEA",

            gridwidth=1,

            zeroline=False,

            ticksuffix="%"

        ),

        yaxis=dict(

            title="",

            categoryorder="array",

            categoryarray=ordem

        ),

        margin=dict(

            l=170,

            r=120,

            t=120,

            b=360

        )

    )

         # =====================================================
    # Painel Executivo
    # =====================================================

    insight = (
        "<b>PRINCIPAIS INSIGHTS</b><br><br>"
        f"• <b>{share.attrs['ganho']}</b> apresentou o maior ganho de participação "
        f"(<b>{share.attrs['ganho_delta']:+.1f} p.p.</b>).<br>"
        f"• <b>{share.attrs['perda']}</b> registrou a maior redução "
        f"(<b>{share.attrs['perda_delta']:+.1f} p.p.</b>).<br>"
        f"• As três maiores companhias "
        f"(<b>{share.attrs['top3_empresas']}</b>) "
        f"concentram aproximadamente "
        f"<b>{share.attrs['top3']:.1f}%</b> do mercado em 2026."
    )

    fig.add_annotation(

        x=0.5,

        y=-0.16,

        xref="paper",

        yref="paper",

        showarrow=False,

        align="left",

        text=insight,

        bordercolor="#D9D9D9",

        borderwidth=1,

        borderpad=12,

        bgcolor="#F8F9FA",

        font=dict(

            size=12

        ),

        width=850

    )

    # =====================================================
    # Rodapé
    # =====================================================

    fig.add_annotation(

        x=1,

        y=-0.28,

        xref="paper",

        yref="paper",

        showarrow=False,

        text="<i>Fonte: ANAC (2019–2026) • Elaboração própria</i>",

        xanchor="right",

        font=dict(

            size=11,

            color="gray"

        )

    )

    hovertemplate=(
    "<b>%{y}</b><br>"
    "Participação: %{x:.1f}%<br>"
    "Passageiros: %{customdata:,.0f}<br>"
    "Período: Jan–Mai<br>"
    "<extra></extra>"
)

    return fig
share_data = market_share_companhias(voos)

fig3 = fig_market_share(share_data)

fig3.show()

## 5. Concentração da malha aérea (jan-mai 2019 x 2026)

**Pergunta de negócio:** o tráfego está mais concentrado em poucos hubs, ou mais distribuído pelo país?

In [11]:
def concentracao_malha_aerea(df: pd.DataFrame, top_n: int = 5) -> pd.DataFrame:
    """Calcula concentração de tráfego nos principais aeroportos de origem.

    Args:
        df: DataFrame de voos regulares.
        top_n: número de aeroportos a destacar.

    Returns:
        DataFrame com ["ANO", "AEROPORTO_DE_ORIGEM_NOME", "PASSAGEIROS_PAGOS", "PCT"].
    """
    janmai = df[(df["MES"] <= 5) & (df["ANO"].isin([2019, 2026]))]
    agg = janmai.groupby(["ANO", "AEROPORTO_DE_ORIGEM_NOME"], as_index=False)["PASSAGEIROS_PAGOS"].sum()
    agg["PCT"] = agg.groupby("ANO")["PASSAGEIROS_PAGOS"].transform(lambda s: s / s.sum() * 100)

    top_aeroportos = agg[agg["ANO"] == 2026].nlargest(top_n, "PCT")["AEROPORTO_DE_ORIGEM_NOME"].tolist()
    return agg[agg["AEROPORTO_DE_ORIGEM_NOME"].isin(top_aeroportos)].sort_values(
        ["ANO", "PCT"], ascending=[True, False]
    )


def fig_concentracao_malha(conc: pd.DataFrame) -> go.Figure:
    """Gera barras comparando concentração de tráfego nos 5 maiores aeroportos, 2019 x 2026."""
    ordem = conc[conc["ANO"] == 2026].sort_values("PCT", ascending=False)["AEROPORTO_DE_ORIGEM_NOME"].tolist()
    fig = go.Figure()
    for ano, cor in [(2019, COR_NEUTRA), (2026, COR_PRINCIPAL)]:
        sub = conc[conc["ANO"] == ano].set_index("AEROPORTO_DE_ORIGEM_NOME").reindex(ordem).reset_index()
        fig.add_trace(go.Bar(
            x=sub["AEROPORTO_DE_ORIGEM_NOME"], y=sub["PCT"], name=str(ano),
            marker_color=cor, text=[f"{v:.1f}%" for v in sub["PCT"]], textposition="outside",
        ))
    top5_2019 = conc[conc["ANO"] == 2019]["PCT"].sum()
    top5_2026 = conc[conc["ANO"] == 2026]["PCT"].sum()
    fig.add_annotation(
        xref="paper", x=1.0, y=1.15, yref="paper", showarrow=False, align="right",
        text=f"Top 5 concentram {top5_2019:.1f}% (2019) x {top5_2026:.1f}% (2026) do tráfego",
        font=dict(size=12, color=COR_PRINCIPAL),
    )
    fig.update_layout(
        template=TEMPLATE, title="Concentração da malha aérea — top 5 aeroportos de origem",
        yaxis_title="% do tráfego (jan-mai)", barmode="group",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
        margin=dict(t=90, b=40, l=60, r=40), height=420,
    )
    return fig


conc = concentracao_malha_aerea(voos)
fig4 = fig_concentracao_malha(conc)
fig4.show()

**Leitura:** a concentração nos 5 maiores aeroportos caiu ligeiramente (48,0% → 46,7%), sugerindo leve interiorização/distribuição do tráfego — não uma tendência forte, mas consistente com expansão de rotas regionais no período.

## 6. Exportação para o README

Gera as 4 imagens estáticas (PNG) para embutir no `README.md` do repositório — GitHub não renderiza HTML interativo do Plotly, então o notebook fica interativo (rodando no Colab/Jupyter) e o README fica com o "print" estático.

Conclusões Executivas

• O mercado brasileiro apresentou recuperação completa após a pandemia.

• A demanda ultrapassou o nível pré-pandemia em 2025.

• O market share permaneceu altamente concentrado nas três maiores companhias.

• Houve pequena redução na concentração da malha aérea, indicando maior distribuição dos fluxos.

• O crescimento observado entre 2023 e 2026 sugere continuidade da expansão do transporte aéreo nacional.

In [27]:
import os
import kaleido
import asyncio

os.makedirs("assets", exist_ok=True)

figs = {
    "01_evolucao_historica.png": fig1,
    "02_recuperacao_pandemia.png": fig2,
    "03_market_share.png": fig3,
    "04_concentracao_malha.png": fig4,
}

async def export_images():
    for nome, fig in figs.items():
        try:
            # O kaleido.write_fig é uma corrotina e precisa ser aguardada no ambiente correto
            img_bytes = await kaleido.write_fig(fig, "png", scale=2, width=900, height=500)
            with open(f"assets/{nome}", "wb") as f:
                f.write(img_bytes)
            print(f"Exportado com sucesso: assets/{nome}")
        except Exception as exc:
            print(f"Falha ao exportar {nome}: {exc}")

# Executa a exportação assíncrona
import asyncio
loop = asyncio.get_event_loop()
if loop.is_running():
    # No Jupyter/Colab o loop já está rodando
    import nest_asyncio
    nest_asyncio.apply()
    asyncio.run(export_images())
else:
    asyncio.run(export_images())

Falha ao exportar 01_evolucao_historica.png: ('The browser seemed to close immediately after starting.', 'You can set the `logging.Logger` level lower to see more output.', 'You may try installing a known working copy of Chrome by running ', '`$ choreo_get_chrome`.It may be your browser auto-updated and will now work upon restart. The browser we tried to start is located at /root/.local/share/choreographer/deps/chrome-linux64/chrome.')
Falha ao exportar 02_recuperacao_pandemia.png: ('The browser seemed to close immediately after starting.', 'You can set the `logging.Logger` level lower to see more output.', 'You may try installing a known working copy of Chrome by running ', '`$ choreo_get_chrome`.It may be your browser auto-updated and will now work upon restart. The browser we tried to start is located at /root/.local/share/choreographer/deps/chrome-linux64/chrome.')
Falha ao exportar 03_market_share.png: ('The browser seemed to close immediately after starting.', 'You can set the `l